# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

tf.random.set_seed(42)
np.random.seed(42)

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

device = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"
print("Using device:", device)

print_every = 700


2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Using device: /GPU:0


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[idxs[i:i+B]], self.y[idxs[i:i+B]]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)


In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(
            channel_1,
            kernel_size=5,
            padding="same",
            activation="relu",
            kernel_initializer=initializer
        )
        self.conv2 = tf.keras.layers.Conv2D(
            channel_2,
            kernel_size=3,
            padding="same",
            activation="relu",
            kernel_initializer=initializer
        )
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(
            num_classes,
            activation="softmax",
            kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        if len(x.shape) == 4 and x.shape[1] == 3 and x.shape[-1] != 3:
            x = tf.transpose(x, [0, 2, 3, 1])
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores


In [7]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def reset_metric(metric):
    if hasattr(metric, "reset_state"):
        metric.reset_state()
    else:
        metric.reset_states()


def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        t = 0
        for epoch in range(num_epochs):
            reset_metric(train_loss)
            reset_metric(train_accuracy)
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    if t % print_every == 0:
                        reset_metric(val_loss)
                        reset_metric(val_accuracy)
                        for test_x, test_y in val_dset:
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)
                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print(template.format(t, epoch + 1,
                                             train_loss.result(),
                                             train_accuracy.result() * 100,
                                             val_loss.result(),
                                             val_accuracy.result() * 100))
                    t += 1
        reset_metric(val_loss)
        reset_metric(val_accuracy)
        for test_x, test_y in val_dset:
            prediction = model(test_x, training=False)
            t_loss = loss_fn(test_y, prediction)
            val_loss.update_state(t_loss)
            val_accuracy.update_state(test_y, prediction)
        print('Final Val Loss: {}, Final Val Accuracy: {}'.format(
            val_loss.result(),
            val_accuracy.result() * 100
        ))
        globals()['last_trained_model'] = model
        globals()['last_val_accuracy'] = float(val_accuracy.result().numpy())
        return model


In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

_ = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 2.9114797115325928, Accuracy: 6.25, Val Loss: 2.8250410556793213, Val Accuracy: 12.700000762939453
Iteration 700, Epoch 1, Loss: 1.8422510623931885, Accuracy: 38.40941619873047, Val Loss: 1.5695396661758423, Val Accuracy: 45.400001525878906
Final Val Loss: 1.774360179901123, Final Val Accuracy: 40.5


Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [10]:
learning_rate = 1e-2
channel_1, channel_2, num_classes = 64, 32, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

_ = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.370880126953125, Accuracy: 4.6875, Val Loss: 29.42048454284668, Val Accuracy: 12.5
Iteration 700, Epoch 1, Loss: 1.7622872591018677, Accuracy: 38.73929977416992, Val Loss: 1.422919511795044, Val Accuracy: 49.70000076293945
Final Val Loss: 1.411772608757019, Final Val Accuracy: 51.30000305175781


# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [11]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

_ = train_part34(model_init_fn, optimizer_init_fn)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.3698201179504395, Accuracy: 4.6875, Val Loss: 2.739375114440918, Val Accuracy: 14.59999942779541
Iteration 700, Epoch 1, Loss: 1.8490798473358154, Accuracy: 38.32471466064453, Val Loss: 1.6602836847305298, Val Accuracy: 43.79999923706055
Final Val Loss: 1.5969675779342651, Final Val Accuracy: 45.89999771118164


Альтернативный менее гибкий способ обучения:

In [12]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 1.8273 - sparse_categorical_accuracy: 0.3875 - val_loss: 1.7250 - val_sparse_categorical_accuracy: 0.4250
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.6819 - sparse_categorical_accuracy: 0.4370


[1.6819390058517456, 0.43700000643730164]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [13]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(64, kernel_size=5, padding="same", activation="relu", kernel_initializer=initializer),
        tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="relu", kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation="softmax", kernel_initializer=initializer)
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 1e-2
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

_ = train_part34(model_init_fn, optimizer_init_fn)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Iteration 0, Epoch 1, Loss: 3.658153772354126, Accuracy: 10.9375, Val Loss: 35.837677001953125, Val Accuracy: 10.5
Iteration 700, Epoch 1, Loss: 1.8719409704208374, Accuracy: 34.28583908081055, Val Loss: 1.6164491176605225, Val Accuracy: 41.20000076293945
Final Val Loss: 1.574604868888855, Final Val Accuracy: 41.900001525878906


In [14]:
model = model_init_fn()
model.compile(optimizer=optimizer_init_fn(),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)


766/766 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 1.7760 - sparse_categorical_accuracy: 0.3871 - val_loss: 1.4012 - val_sparse_categorical_accuracy: 0.5050
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.3925 - sparse_categorical_accuracy: 0.5024


[1.3924522399902344, 0.5023999810218811]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [15]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [16]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

_ = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.218406915664673, Accuracy: 9.375, Val Loss: 2.877981185913086, Val Accuracy: 12.100000381469727
Iteration 700, Epoch 1, Loss: 1.8297220468521118, Accuracy: 38.770503997802734, Val Loss: 1.6291474103927612, Val Accuracy: 44.400001525878906
Final Val Loss: 1.6588544845581055, Final Val Accuracy: 43.70000076293945


Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.keras.initializers.HeNormal()
        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D()
        self.drop1 = tf.keras.layers.Dropout(0.25)
        self.conv3 = tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.conv4 = tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D()
        self.drop2 = tf.keras.layers.Dropout(0.30)
        self.conv5 = tf.keras.layers.Conv2D(128, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn5 = tf.keras.layers.BatchNormalization()
        self.conv6 = tf.keras.layers.Conv2D(128, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn6 = tf.keras.layers.BatchNormalization()
        self.pool3 = tf.keras.layers.MaxPooling2D()
        self.drop3 = tf.keras.layers.Dropout(0.40)
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(256, use_bias=False, kernel_initializer=initializer)
        self.bn7 = tf.keras.layers.BatchNormalization()
        self.drop4 = tf.keras.layers.Dropout(0.50)
        self.fc2 = tf.keras.layers.Dense(10, activation="softmax")

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)
        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)
        x = self.conv5(x)
        x = self.bn5(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv6(x)
        x = self.bn6(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool3(x)
        x = self.drop3(x, training=training)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn7(x, training=training)
        x = tf.nn.relu(x)
        x = self.drop4(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate=learning_rate)

custom_model = train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)
custom_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=[tf.keras.metrics.sparse_categorical_accuracy]
)
custom_model.evaluate(X_test, y_test, batch_size=64)


Iteration 0, Epoch 1, Loss: 2.9654998779296875, Accuracy: 9.375, Val Loss: 3.720475435256958, Val Accuracy: 9.199999809265137
Iteration 700, Epoch 1, Loss: 1.5856359004974365, Accuracy: 43.44463348388672, Val Loss: 1.142556071281433, Val Accuracy: 59.10000228881836
Iteration 1400, Epoch 2, Loss: 1.0867977142333984, Accuracy: 61.301673889160156, Val Loss: 0.9609590768814087, Val Accuracy: 66.9000015258789
Iteration 2100, Epoch 3, Loss: 0.9055342078208923, Accuracy: 68.00582122802734, Val Loss: 1.0988681316375732, Val Accuracy: 63.400001525878906
Iteration 2800, Epoch 4, Loss: 0.8189294338226318, Accuracy: 71.2692642211914, Val Loss: 0.7518402338027954, Val Accuracy: 73.5999984741211
Iteration 3500, Epoch 5, Loss: 0.7459566593170166, Accuracy: 73.69851684570312, Val Loss: 0.8449718356132507, Val Accuracy: 70.20000457763672
Iteration 4200, Epoch 6, Loss: 0.6856688857078552, Accuracy: 76.07395935058594, Val Loss: 0.6693297028541565, Val Accuracy: 77.20000457763672
Iteration 4900, Epoch 7, 

[0.5412082076072693, 0.8159999847412109]

Опишите все эксперименты, результаты. Сделайте выводы.

В лабораторной работе были реализованы и протестированы несколько моделей классификации изображений CIFAR-10 с использованием TensorFlow и Keras. Сначала была проверена простая двухслойная полносвязная сеть, реализованная через Model Subclassing API. Она показала сравнительно невысокое качество и достигла около 40.5% accuracy на валидационной выборке после одной эпохи обучения. Далее была реализована трехслойная сверточная нейронная сеть, состоящая из двух сверточных слоев и полносвязного выходного слоя. При обучении с помощью SGD с Nesterov momentum = 0.9 эта модель достигла 51.3% accuracy на валидационной выборке уже после одной эпохи, то есть выполнила требование задания.

После этого аналогичная архитектура была собрана с использованием Sequential API и обучена двумя способами: через собственный цикл обучения и через встроенный метод model.fit. В собственном цикле обучения модель показала 41.9% accuracy на валидационной выборке, а при использовании model.fit достигла 50.5% на валидации и 50.24% на тестовой выборке. Также была протестирована полносвязная сеть, реализованная через Functional API, которая показала около 43.7% accuracy. В заключительной части была построена более глубокая сверточная сеть с Batch Normalization, MaxPooling, Dropout и оптимизатором Adam. Эта модель показала наилучший результат: 83.0% accuracy на валидационной выборке и 81.6% на тестовой выборке за 10 эпох обучения, что значительно превышает требуемый в задании порог 70%.

По итогам экспериментов можно сделать вывод, что для задачи классификации изображений CIFAR-10 сверточные нейронные сети работают заметно лучше, чем простые полносвязные модели, так как лучше учитывают пространственную структуру изображений. Также результаты показывают, что усложнение архитектуры, добавление нормализации, pooling-слоев и dropout, а также использование оптимизатора Adam позволяют существенно повысить качество классификации и сделать обучение более устойчивым.